# GRAST — One bounded easy SFT recovery round
Chạy **một vòng 1.000 updates** từ selected2200; chỉ train toán một phép +/−. Đây là bước chuẩn bị trước WP09, chưa triển khai confidence policy.

1. Chọn GPU trong **Runtime → Change runtime type** (T4 được hỗ trợ).
2. Upload kit ZIP và checkpoint `selected_model.pt` vào `MyDrive/GRAST/easy_recovery/inputs/`.
3. Chạy với `SMOKE_ONLY = True` để kiểm tra 2 updates. Sau khi smoke qua, đổi thành `False` và chạy cell cấu hình cùng các cell bên dưới để train đầy đủ.
4. Mọi tiến độ đều in trực tiếp. Không chỉnh recipe theo kết quả gate; kết thúc vòng này rồi chuyển WP09.

16 câu selection chỉ dùng chọn checkpoint, 48 câu gate chỉ chấm sau khi chọn xong. Hai nhóm không trùng cặp toán hạng với nhau hoặc các phép toán con đã biết trong train/selection/pilot cũ. Cả hai thuộc **development validation**, từng được dùng đo loss; không phải test chưa từng đụng tới. Không mở test set.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import importlib.util
import json
import os
import signal
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
LAST_RUN_DIRECTORY = None


def stream(command, cwd=None):
    global LAST_RUN_DIRECTORY
    print("Running:", " ".join(map(str, command)), flush=True)
    with subprocess.Popen(
        list(map(str, command)),
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                try:
                    record = json.loads(line)
                except (ValueError, TypeError):
                    continue
                if record.get("event") in ("easy_recovery_started", "easy_recovery_finished"):
                    LAST_RUN_DIRECTORY = record["run_directory"]
            returncode = process.wait()
        except KeyboardInterrupt:
            print("Stopping child process; resume from latest committed checkpoint.", flush=True)
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        if LAST_RUN_DIRECTORY:
            print("Last run directory:", LAST_RUN_DIRECTORY, flush=True)
        raise RuntimeError(f"Command failed with exit code {returncode}; inspect logs above.")


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
INPUT_ROOT = Path("/content/drive/MyDrive/GRAST/easy_recovery/inputs")
KIT = INPUT_ROOT / "grast-easy-recovery-kit-20261127.zip"
SOURCE_CHECKPOINT = INPUT_ROOT / "selected_model.pt"
KIT_SHA256 = "c8b15e13ab936af43a7efc649fedc1191f210fa4d9a097c95bc63bb68ab606cf"
SOURCE_SHA256 = "d2ea45abc8f81b516552c184d75c595b95fa2bd1abdc260538e497da4b627dc8"
SMOKE_ONLY = True
RESUME_CHECKPOINT = None  # Full run: set to the absolute Drive path of checkpoints/latest.pt.
OUTPUT_ROOT = Path("/content/drive/MyDrive/GRAST/easy_recovery") / (
    "smoke_runs" if SMOKE_ONLY else "runs"
)

if SMOKE_ONLY and RESUME_CHECKPOINT is not None:
    raise ValueError("Smoke starts fresh; set SMOKE_ONLY=False to resume a full run.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not KIT.is_file() or not SOURCE_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Upload kit and selected_model.pt into {INPUT_ROOT}")
if sha256(KIT) != KIT_SHA256 or sha256(SOURCE_CHECKPOINT) != SOURCE_SHA256:
    raise ValueError("Input checksum mismatch; use the original kit and selected2200 export.")
print(
    json.dumps(
        {"event": "inputs_verified", "kit_sha256": KIT_SHA256, "source_sha256": SOURCE_SHA256}
    ),
    flush=True,
)

In [ ]:
import zipfile

EXPECTED_FILES = {
    "data/manifest.json",
    "data/train.jsonl",
    "data/selection.jsonl",
    "data/gate.jsonl",
    "tokenizer/vocab.json",
    "tokenizer/merges.json",
    "tokenizer/tokenizer_manifest.json",
    "tokenizer/tokenizer_config.json",
    "easy_recovery.toml",
    "grast-0.1.0-py3-none-any.whl",
    "bundle_manifest.json",
}
KIT_ROOT = Path(tempfile.mkdtemp(prefix="grast-easy-kit-"))
with zipfile.ZipFile(KIT) as archive:
    infos, names = archive.infolist(), archive.namelist()
    if (
        set(names) != EXPECTED_FILES
        or len(names) != len(set(names))
        or sum(i.file_size for i in infos) > 256 * 1024**2
        or any((i.external_attr >> 16) & 0o170000 == 0o120000 for i in infos)
    ):
        raise ValueError("Unsafe or unexpected kit entries")
    archive.extractall(KIT_ROOT)
manifest = json.loads((KIT_ROOT / "bundle_manifest.json").read_text())
if set(manifest["files"]) != EXPECTED_FILES - {"bundle_manifest.json"}:
    raise ValueError("Kit manifest entries mismatch")
for name, digest in manifest["files"].items():
    if sha256(KIT_ROOT / name) != digest:
        raise ValueError(f"Kit payload mismatch: {name}")
# Install exactly the supplied wheel; keep the Colab PyTorch installation.
stream(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        KIT_ROOT / "grast-0.1.0-py3-none-any.whl",
    ]
)
missing = [
    package for package in ("numpy", "matplotlib") if importlib.util.find_spec(package) is None
]
if missing:
    stream([sys.executable, "-m", "pip", "install", *missing])
stream(
    [
        sys.executable,
        "-u",
        "-c",
        (
            "import sys; from llm_scratch.easy_recovery.bundle import verify; "
            "print(verify(sys.argv[1]), flush=True)"
        ),
        KIT_ROOT,
    ]
)

In [ ]:
import torch

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("This package requires Python 3.11–3.13.")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime, then restart the notebook.")
PRECISION = (
    "bfloat16"
    if torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
    else "float16"
)
if RESUME_CHECKPOINT is not None:
    saved = torch.load(RESUME_CHECKPOINT, map_location="cpu", weights_only=True, mmap=True)
    PRECISION = saved["state"]["training_config"]["precision"]
    print(
        json.dumps(
            {
                "event": "resume_requested",
                "step": saved["state"]["step"],
                "saved_runtime": saved["state"]["runtime"],
            }
        ),
        flush=True,
    )
    del saved
print(
    json.dumps(
        {
            "event": "runtime_ready",
            "python": sys.version,
            "pytorch": torch.__version__,
            "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0),
            "precision": PRECISION,
        }
    ),
    flush=True,
)
# Exact resume requires the same kit, GPU/runtime and precision as the saved run.
# The runner rejects incompatible resumes rather than silently warm-starting.

## Training — live logs
`SMOKE_ONLY=True` dừng ở update 2, lưu full checkpoint và **không chấm gate**. Full run khởi tạo optimizer/scheduler/scaler/RNG/cursor mới từ weights selected2200. Để tiếp tục sau ngắt kết nối, giữ nguyên runtime/kit, đặt `SMOKE_ONLY=False` và `RESUME_CHECKPOINT` thành đường dẫn `checkpoints/latest.pt` được in bên dưới. Truyền lại checkpoint nguồn để xác minh lineage và chấm baseline cuối vòng.

Checkpoint selection: tối đa macro verified EM theo hai nguồn, sau đó macro final EM, giữ bước sớm nhất khi hòa; step 0 là ứng viên hợp lệ. Trace verification ở đây là khớp rationale chuẩn hóa khoảng trắng, không phải bộ chấm reasoning tổng quát.

In [ ]:
LAST_RUN_DIRECTORY = None
command = [
    sys.executable,
    "-u",
    "-m",
    "llm_scratch.easy_recovery.runner",
    "--config",
    KIT_ROOT / "easy_recovery.toml",
    "--prepared-root",
    KIT_ROOT / "data",
    "--tokenizer",
    KIT_ROOT / "tokenizer",
    "--source-checkpoint",
    SOURCE_CHECKPOINT,
    "--device",
    "cuda",
    "--precision",
    PRECISION,
    "--output-root",
    OUTPUT_ROOT,
]
if SMOKE_ONLY:
    command += ["--stop-after", "2"]
if RESUME_CHECKPOINT is not None:
    command += ["--resume", RESUME_CHECKPOINT]
stream(command)
if LAST_RUN_DIRECTORY is None:
    raise RuntimeError("Runner did not return an exact run directory; inspect logs above.")
RUN_DIRECTORY = Path(LAST_RUN_DIRECTORY)
report = json.loads((RUN_DIRECTORY / "artifacts/recovery_report.json").read_text())
if SMOKE_ONLY and (
    report["status"] != "controlled_stop" or report["step"] != 2 or report["gate_evaluated"]
):
    raise RuntimeError("Smoke acceptance failed")
if not SMOKE_ONLY and (report["status"] != "completed" or not report["gate_evaluated"]):
    raise RuntimeError("Full run has not finished and scored the frozen gate")
print(json.dumps(report, indent=2), flush=True)
print("Resume checkpoint:", RUN_DIRECTORY / "checkpoints/latest.pt", flush=True)
print("Inference export:", RUN_DIRECTORY / "checkpoints/selected_model.pt", flush=True)
(RUN_DIRECTORY / "artifacts/colab_inputs.json").write_text(
    json.dumps(
        {"kit_sha256": KIT_SHA256, "kit_manifest": manifest, "source_sha256": SOURCE_SHA256},
        indent=2,
    )
    + "\n"
)

## Download kết quả
Summary ZIP loại mọi weights/full checkpoint. Model được tải riêng. Full checkpoint đã nằm trên Drive để resume; không dùng inference export để tiếp tục optimizer. Gửi lại **summary ZIP + selected_model.pt** sau full run để chốt kết quả và chuẩn bị sample bank mới cho WP09. Smoke không phải bằng chứng model đã giải toán tốt.

In [ ]:
from google.colab import files

SUMMARY_ZIP = RUN_DIRECTORY.parent / (RUN_DIRECTORY.name + "-summary.zip")
with zipfile.ZipFile(SUMMARY_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIRECTORY.rglob("*")):
        if path.is_file() and path.suffix.lower() not in (".pt", ".pth", ".ckpt", ".safetensors"):
            archive.write(path, path.relative_to(RUN_DIRECTORY).as_posix())
print(
    json.dumps(
        {
            "summary": str(SUMMARY_ZIP),
            "sha256": sha256(SUMMARY_ZIP),
            "selected_model_sha256": sha256(RUN_DIRECTORY / "checkpoints/selected_model.pt"),
        }
    ),
    flush=True,
)
files.download(str(SUMMARY_ZIP))
if not SMOKE_ONLY:
    files.download(str(RUN_DIRECTORY / "checkpoints/selected_model.pt"))